In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!git clone https://github.com/facebookresearch/detectron2.git

Cloning into 'detectron2'...
remote: Enumerating objects: 15297, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 15297 (delta 7), reused 7 (delta 1), pack-reused 15275
Receiving objects: 100% (15297/15297), 6.18 MiB | 19.13 MiB/s, done.
Resolving deltas: 100% (11121/11121), done.


In [3]:
%cd detectron2

/content/detectron2


In [4]:
!pip install -e .

Obtaining file:///content/detectron2
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 19.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 17.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61400 sha256=38a1bd7cb05e5d88c31b913a6eac779a69b3e9e2627e077e6196e5620387a5f1
  Stored in directory: /root/.cache/pip/wheels/01/c0/af/77c1cf53a1be9e42a52b48e5af2169d40ec2e89f7362489dd0
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144554 sha256=913045fcdf57443577dac5d6bf60ef1a54e718580655477e

In [5]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
import os
import multiprocessing as mp
import json

In [6]:
image_folder = "/content/drive/MyDrive/xBD dataset/xbd/train/images"
labels_folder = "/content/drive/MyDrive/xBD dataset/xbd/train/labels2"


In [7]:
# annotations =

In [8]:
# # Combine all JSON files into a single list
# annotations = []
# for json_file in os.listdir(labels_folder):
#     if json_file.endswith(".json"):
#         with open(os.path.join(labels_folder, json_file), 'r') as f:
#             annotations.extend(json.load(f))

In [9]:
# Register dataset
dataset_name = "xbd_train"
register_coco_instances(dataset_name, {}, labels_folder, image_folder)


In [10]:
# Set up metadata
metadata = MetadataCatalog.get(dataset_name)
metadata.set(thing_classes=[])

namespace(name='xbd_train',
          json_file='/content/drive/MyDrive/xBD dataset/xbd/train/labels2',
          image_root='/content/drive/MyDrive/xBD dataset/xbd/train/images',
          evaluator_type='coco',
          thing_classes=[])

In [11]:
# Configuration
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml"))

cfg.DATASETS.TRAIN = (dataset_name,)
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml")
cfg.SOLVER.BASE_LR = 0.001
cfg.SOLVER.MAX_ITER = 600
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.DATALOADER.NUM_WORKERS = mp.cpu_count()
cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 1  # Assuming you only have one class for semantic segmentation
cfg.OUTPUT_DIR = "outputModels"
cfg.MODEL.DEVICE = "cuda"

In [12]:
# Create output directory
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

In [13]:
# Train the model
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(True)
trainer.train()

[01/03 07:28:41 d2.engine.defaults]: Model:
PanopticFPN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2): 

IsADirectoryError: ignored